# Binoculars on Google Colab

This notebook installs and runs the original Binoculars repository with its default Falcon-7B + Falcon-7B-Instruct model pair.

For the closest match to the published setup:
- In Colab, choose `Runtime` -> `Change runtime type` -> `GPU` before running.
- If you hit an out-of-memory error, reconnect to a larger GPU runtime and rerun from the top.
- The first model load can take a while because the Falcon weights are large.

Note: current Colab runtimes use Python 3.12, and the repo's old `transformers` pin can fail there while building `tokenizers`. This notebook installs a current compatible `transformers` stack first, then installs Binoculars itself without reusing the old dependency pin.

In [1]:
import os

REPO_URL = "https://github.com/ahans30/Binoculars.git"
REPO_DIR = "/content/Binoculars"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/Binoculars

!pip -q install -U pip
!pip -q install -U setuptools wheel
!pip -q install "jedi>=0.19" "transformers>=4.46,<5" "tokenizers>=0.20" accelerate sentencepiece datasets numpy gradio gradio_client scikit-learn seaborn pandas huggingface_hub[hf_xet]
!pip -q install -e . --no-deps


Cloning into '/content/Binoculars'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 208 (delta 72), reused 64 (delta 64), pack-reused 119 (from 1)
Receiving objects: 100% (208/208), 54.08 MiB | 11.42 MiB/s, done.
Resolving deltas: 100% (102/102), done.
Updating files: 100% (27/27), done.
/content/Binoculars
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 89.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for Binoculars (pyproject.toml) ... done


In [2]:
import os
import platform
import shutil
import subprocess
import torch

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("Loaded HF_TOKEN from Colab secrets.")
except Exception:
    pass

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "This Colab runtime is CPU-only. Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU, then rerun from the top."
    )

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)


Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
CUDA device count: 1
GPU: NVIDIA A100-SXM4-80GB


## Evaluate HC3 CSVs

Upload `evaluate_samples.py` plus one or more HC3 CSVs created by `create_hc3_sample.py`, such as `hc3_unified_1000_seed42.csv` or any `hc3_<source>_200_seed42.csv`.

Each CSV must use the HC3 wide schema: `hc3_row_id`, `source`, `question`, `human_answers`, and `chatgpt_answers`. The cell expands those answer-list columns into row-wise `text` and `label` samples before running Binoculars.

In [4]:
from google.colab import files
from pathlib import Path
import ast
import json
import os
import re
import pandas as pd
import subprocess
import sys

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

script_uploaded_name = None
csv_uploaded_names = []

for filename in uploaded.keys():
    if filename.endswith(".py"):
        script_uploaded_name = filename
    elif filename.endswith(".csv"):
        csv_uploaded_names.append(filename)

if not script_uploaded_name:
    raise FileNotFoundError("Upload evaluate_samples.py along with the HC3 CSV file(s).")
if not csv_uploaded_names:
    raise FileNotFoundError("Upload at least one HC3 CSV file created by create_hc3_sample.py.")

script_destination_path = os.path.join(REPO_DIR, "evaluate_samples.py")
with open(script_destination_path, "wb") as f:
    f.write(uploaded[script_uploaded_name])
print(f"Saved '{script_uploaded_name}' as 'evaluate_samples.py' to '{script_destination_path}'")

os.chdir(REPO_DIR)

raw_input_dir = Path(REPO_DIR) / "hc3_uploaded_inputs"
raw_input_dir.mkdir(exist_ok=True)

def safe_stem(filename):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(filename).stem)

def parse_answer_list(value):
    if isinstance(value, list):
        parsed = value
    elif isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            try:
                parsed = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                parsed = [value]
    elif pd.isna(value):
        return []
    else:
        parsed = value

    if isinstance(parsed, list):
        return [str(item).strip() for item in parsed if pd.notna(item) and str(item).strip()]
    if pd.notna(parsed) and str(parsed).strip():
        return [str(parsed).strip()]
    return []

def build_hc3_eval_dataframe(df):
    required_columns = {"hc3_row_id", "source", "question", "human_answers", "chatgpt_answers"}
    missing_columns = sorted(required_columns.difference(df.columns))
    if missing_columns:
        raise ValueError(
            "Unsupported CSV format. Expected an HC3 CSV created by create_hc3_sample.py. "
            f"Missing columns: {missing_columns}"
        )

    rows = []
    for row in df.itertuples(index=False):
        base = {
            "hc3_id": row.hc3_row_id,
            "hc3_row_id": row.hc3_row_id,
            "source": row.source,
            "question": row.question,
        }
        for column_name, label in [("human_answers", "human"), ("chatgpt_answers", "ai")]:
            for answer_index, text in enumerate(parse_answer_list(getattr(row, column_name))):
                rows.append({
                    **base,
                    "answer_type": label,
                    "answer_index": answer_index,
                    "text": text,
                    "label": label,
                })

    eval_df = pd.DataFrame(rows)
    if eval_df.empty:
        raise ValueError("The uploaded HC3 CSV did not contain any answer text to score.")
    return eval_df

def merge_scores(eval_df, scored_df):
    if "sample_id" in scored_df.columns:
        return pd.merge(eval_df, scored_df, on="sample_id", how="left", validate="one_to_one", suffixes=("", "_scored"))
    if len(scored_df) == len(eval_df):
        scored_trimmed = scored_df.drop(columns=[col for col in ["text", "label"] if col in scored_df.columns])
        return pd.concat([eval_df.reset_index(drop=True), scored_trimmed.reset_index(drop=True)], axis=1)

    eval_with_index = eval_df.copy()
    scored_with_index = scored_df.copy()
    eval_with_index["_merge_idx"] = eval_with_index.groupby(["text", "label"]).cumcount()
    scored_with_index["_merge_idx"] = scored_with_index.groupby(["text", "label"]).cumcount()
    return pd.merge(
        eval_with_index,
        scored_with_index,
        on=["text", "label", "_merge_idx"],
        how="left",
        validate="one_to_one",
        suffixes=("", "_scored"),
    ).drop(columns=["_merge_idx"])

all_results = {}

for csv_uploaded_name in csv_uploaded_names:
    run_name = safe_stem(csv_uploaded_name)
    print()
    print(f"Evaluating {csv_uploaded_name}")

    raw_input_path = raw_input_dir / f"{run_name}.csv"
    with open(raw_input_path, "wb") as f:
        f.write(uploaded[csv_uploaded_name])

    original_df = pd.read_csv(raw_input_path)
    eval_df = build_hc3_eval_dataframe(original_df).reset_index(drop=True)
    eval_df["sample_id"] = range(len(eval_df))

    data_input_path = Path(REPO_DIR) / f"{run_name}_binoculars_input.csv"
    output_dir = f"{run_name}_eval"
    eval_df.to_csv(data_input_path, index=False)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Prepared {len(eval_df)} evaluation rows in '{data_input_path}'")

    cmd = [
        sys.executable, "evaluate_samples.py",
        "--input", str(data_input_path),
        "--text-column", "text",
        "--label-column", "label",
        "--mode", "low-fpr",
        "--output-dir", output_dir,
    ]
    subprocess.run(cmd, check=True)

    with open(Path(output_dir) / "summary.json", "r") as f:
        summary = json.load(f)

    print("Summary:")
    print(json.dumps(summary, indent=2))

    scored_df = pd.read_csv(Path(output_dir) / "scored_samples.csv")
    result_df = merge_scores(eval_df, scored_df)
    all_results[run_name] = result_df

    summary_columns = [
        column for column in [
            "hc3_id", "hc3_row_id", "source", "question", "answer_type", "answer_index", "label", "score", "prediction_text"
        ] if column in result_df.columns
    ]
    display(result_df[summary_columns])

    label_num = pd.to_numeric(result_df["label"].replace({"human": 0, "ai": 1}), errors="coerce")
    prediction_num = pd.to_numeric(result_df["prediction"], errors="coerce")
    mistakes = result_df[label_num.ne(prediction_num)]

    print("Mistakes:")
    mistake_columns = [
        column for column in [
            "hc3_id", "hc3_row_id", "source", "question", "answer_type", "answer_index", "label", "score", "prediction_text", "text"
        ] if column in mistakes.columns
    ]
    display(mistakes[mistake_columns])

print()
print("Finished evaluating:", list(all_results.keys()))


Saving evaluate_samples.py to evaluate_samples.py
Saving hc3_unified_1000_seed42.csv to hc3_unified_1000_seed42 (1).csv
Uploaded: ['evaluate_samples.py', 'hc3_unified_1000_seed42 (1).csv']
Saved 'evaluate_samples.py' as 'evaluate_samples.py' to '/content/Binoculars/evaluate_samples.py'

Evaluating hc3_unified_1000_seed42 (1).csv
Prepared 2836 evaluation rows in '/content/Binoculars/hc3_unified_1000_seed42_1__binoculars_input.csv'
Summary:
{
  "count": 2836,
  "mode": "low-fpr",
  "threshold": 0.8536432310785527,
  "accuracy": 0.9619181946403385,
  "precision": 0.935124508519004,
  "recall": 0.9937325905292479,
  "f1": 0.9635381498987171,
  "f1_at_1pct_tpr": {
    "f1": 0.02067539627842867,
    "threshold": 0.4549180269251333,
    "target_tpr": 0.01,
    "actual_tpr": 0.010445682451253482
  },
  "auc_roc": 0.9922970553123757,
  "confusion_matrix": [
    [
      1301,
      99
    ],
    [
      9,
      1427
    ]
  ],
  "confusion_matrix_labels": [
    "human",
    "ai"
  ],
  "true_ne

,hc3_id,hc3_row_id,source,question,answer_type,answer_index,label,score,prediction_text
0,19149,19149,finance,T-mobile stock: difference between TMUSP vs TMUS,human,0,human,0.984211,Most likely human-written
1,19149,19149,finance,T-mobile stock: difference between TMUSP vs TMUS,ai,0,ai,0.673307,Most likely AI-generated
2,19155,19155,finance,Is there any US bank that does not charge for ...,human,0,human,0.894472,Most likely human-written
3,19155,19155,finance,Is there any US bank that does not charge for ...,ai,0,ai,0.546980,Most likely AI-generated
4,19158,19158,finance,Unemployment Insurance Through Options,human,0,human,1.058036,Most likely human-written
...,...,...,...,...,...,...,...,...,...
2831,19127,19127,wiki_csai,"Please explain what is ""Scientific computing""",ai,0,ai,0.666667,Most likely AI-generated
2832,19138,19138,wiki_csai,"Please explain what is ""BBC Model B""",human,0,human,0.956522,Most likely human-written
2833,19138,19138,wiki_csai,"Please explain what is ""BBC Model B""",ai,0,ai,0.764192,Most likely AI-generated
2834,19139,19139,wiki_csai,"Please explain what is ""O level""",human,0,human,1.046053,Most likely human-written


Mistakes:


/tmp/ipykernel_1912/521460367.py:166: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  label_num = pd.to_numeric(result_df["label"].replace({"human": 0, "ai": 1}), errors="coerce")


,hc3_id,hc3_row_id,source,question,answer_type,answer_index,label,score,prediction_text,text
155,20363,20363,finance,Buying shares in employer's company during IPO,ai,0,ai,1.100592,Most likely human-written,!Your authentication token has expired. Please...
204,20830,20830,finance,Can I deduct my individual Health Insurance Pr...,human,0,human,0.787736,Most likely AI-generated,"Yes, you can. See the instructions for line 29..."
221,20953,20953,finance,I own ASPIRO shares (Jay Z's new company). Now...,ai,1,ai,1.264865,Most likely human-written,chat.openai.comChecking if the site connection...
245,21289,21289,finance,How to work around the Owner Occupancy Affidav...,human,0,human,0.852792,Most likely AI-generated,"Look into the definition of ""primary residence..."
414,22943,22943,finance,Is Volvo a public company?,human,0,human,0.798817,Most likely AI-generated,"There are two different companies named ""Volvo..."
...,...,...,...,...,...,...,...,...,...,...
2680,18797,18797,wiki_csai,"Please explain what is ""Crossover (genetic alg...",human,0,human,0.852761,Most likely AI-generated,In genetic algorithms and evolutionary computa...
2682,18800,18800,wiki_csai,"Please explain what is ""Distributed artificial...",human,0,human,0.787500,Most likely AI-generated,Distributed Artificial Intelligence (DAI) also...
2748,18948,18948,wiki_csai,"Please explain what is ""Statistical hypothesis...",human,0,human,0.745763,Most likely AI-generated,A statistical hypothesis test is a method of s...
2822,19118,19118,wiki_csai,"Please explain what is ""Applied mathematics""",human,0,human,0.835294,Most likely AI-generated,Applied mathematics is the application of math...



Finished evaluating: ['hc3_unified_1000_seed42_1_']


In [5]:
from pathlib import Path
import json

EXPERIMENT_NAME = "Binoculars HC3 evaluation"
DETECTION_METHOD = "Binoculars"
MODEL_USED = "tiiuae/falcon-7b + tiiuae/falcon-7b-instruct"
ADDITIONAL_MODELS_USED = ["tiiuae/falcon-7b", "tiiuae/falcon-7b-instruct"]
NOTES = "Zero-shot Binoculars detector using the configured threshold mode."

def metric_value(summary, *keys):
    for key in keys:
        value = summary.get(key)
        if isinstance(value, dict):
            value = value.get("f1")
        if value is not None:
            return float(value)
    return None

def build_experiment_report(run_name, summary):
    return {
        "experiment_name": EXPERIMENT_NAME,
        "detection_method": DETECTION_METHOD,
        "model_used": MODEL_USED,
        "dataset_used": run_name,
        "num_samples": int(summary.get("count", 0)),
        "additional_details": {
            "additional_models_used": ADDITIONAL_MODELS_USED,
            "notes": NOTES,
        },
        "metrics": {
            "f1_at_1pct_tpr": metric_value(summary, "f1_at_1pct_tpr", "F1@1%TPR"),
            "accuracy": metric_value(summary, "accuracy", "Accuracy"),
            "precision": metric_value(summary, "precision", "precision_ai", "Precision"),
            "recall": metric_value(summary, "recall", "recall_ai", "Recall"),
            "auc_roc": metric_value(summary, "auc_roc", "roc_auc", "AUC-ROC"),
        },
    }

if "csv_uploaded_names" in globals():
    run_names = [safe_stem(filename) for filename in csv_uploaded_names]
else:
    run_names = [path.name.removesuffix("_eval") for path in Path(REPO_DIR).glob("*_eval")]

reports = []
for run_name in run_names:
    summary_path = Path(REPO_DIR) / f"{run_name}_eval" / "summary.json"
    if not summary_path.exists():
        print(f"Skipping {run_name}: missing {summary_path}")
        continue

    with open(summary_path, "r") as f:
        summary = json.load(f)

    report = build_experiment_report(run_name, summary)
    reports.append(report)

    report_path = Path(REPO_DIR) / f"experiment_report_{run_name}.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"Wrote {report_path}")

combined_path = Path(REPO_DIR) / "experiment_reports.json"
with open(combined_path, "w") as f:
    json.dump(reports[0] if len(reports) == 1 else reports, f, indent=2)
print(f"Wrote {combined_path}")

display(reports[0] if len(reports) == 1 else reports)


Wrote /content/Binoculars/experiment_report_hc3_unified_1000_seed42_1_.json
Wrote /content/Binoculars/experiment_reports.json


{'experiment_name': 'Binoculars HC3 evaluation',
 'detection_method': 'Binoculars',
 'model_used': 'tiiuae/falcon-7b + tiiuae/falcon-7b-instruct',
 'dataset_used': 'hc3_unified_1000_seed42_1_',
 'num_samples': 2836,
 'additional_details': {'additional_models_used': ['tiiuae/falcon-7b',
   'tiiuae/falcon-7b-instruct'],
  'notes': 'Zero-shot Binoculars detector using the configured threshold mode.'},
 'metrics': {'f1_at_1pct_tpr': 0.02067539627842867,
  'accuracy': 0.9619181946403385,
  'precision': 0.935124508519004,
  'recall': 0.9937325905292479,
  'auc_roc': 0.9922970553123757}}